In [1]:
import pyspark
from pyspark.sql import SparkSession
from sqlalchemy import create_engine
from dotenv import load_dotenv
from datetime import datetime
import logging
import os
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [2]:
import pandas as pd
import sqlalchemy
from dotenv import load_dotenv
import os

In [3]:
# Create SparkSession
spark = SparkSession \
        .builder \
        .appName("PySpark Exercise Week 6") \
        .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
        .getOrCreate()

In [4]:
logging.basicConfig(
    filename="/home/jovyan/work/log/info.log",
    level=logging.INFO,
    format="[%(asctime)s] [%(levelname)s] %(message)s"
)

In [5]:
SOURCE_URL  = "jdbc:postgresql://source_db:5432/source"
DB_PROPERTIES = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

In [21]:
def load_log_msg(spark: SparkSession, log_msg):

    LOG_DB_URL = "jdbc:postgresql://log_db:5432/log_db"

    LOG_DB_PROPERTIES = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
    }

    table_name = "etl_log"

    # # set config
    # connection_properties = {
    #     "user": LOG_DB_USER,
    #     "password": LOG_DB_PASS,
    #     "driver": "org.postgresql.Driver" # set driver postgres
    # }

    log_msg.write.jdbc(url = LOG_DB_URL,
                  table = table_name,
                  mode = "append",
                  properties = LOG_DB_PROPERTIES)
    
    # print(f"finished writing log..")

In [35]:
def extract_from_db(spark: SparkSession, table_name: str):
    """extract a table from source_db via jdbc"""
    current_timestamp = datetime.now()
    try:
        print(f"Extracting table: {table_name}")
        
        df = spark.read.jdbc(
            url=SOURCE_URL,
            table=table_name,
            properties=DB_PROPERTIES
        )
        print(f"Finished extracting data from table {table_name}: {df.count()} rows, {len(df.columns)} columns")

        log_message = spark.sparkContext \
            .parallelize([("extraction", "source", "success", table_name, current_timestamp)]) \
            .toDF(["step", "component", "status", "table_name", "etl_date"])
        
        return df
        
    except Exception as e:
        print(f"Error extracting data from table {table_name}: {e}")

        log_message = spark.sparkContext \
            .parallelize([("extraction", "source", "failed", table_name, current_timestamp, str(e))]) \
            .toDF(["step", "component", "status", "table_name", "etl_date", "error_msg"])

    finally:
        load_log_msg(spark=spark, log_msg=log_message)
        


df = spark.read \
    .jdbc(url=SOURCE_URL, table="marital_status", properties=DB_PROPERTIES)

In [23]:
df_marital_status = extract_from_db(spark = spark, table_name = "marital_status")
df_education_status = extract_from_db(spark = spark, table_name = "education_status")
df_marketing_campaign_deposit = extract_from_db(spark = spark, table_name = "marketing_campaign_deposit")

Extracting table: marital_status
Finished extracting data from table marital_status: 3 rows, 4 columns
+----------+---------+-------+--------------+--------------------+
|      step|component| status|    table_name|            etl_date|
+----------+---------+-------+--------------+--------------------+
|extraction|   source|success|marital_status|2026-05-30 07:27:...|
+----------+---------+-------+--------------+--------------------+

finished writing log..


In [11]:
df_marital_status.show()

+----------+--------+--------------------+--------------------+
|marital_id|   value|          created_at|          updated_at|
+----------+--------+--------------------+--------------------+
|         1| married|2025-02-28 15:31:...|2025-02-28 15:31:...|
|         2|  single|2025-02-28 15:31:...|2025-02-28 15:31:...|
|         3|divorced|2025-02-28 15:31:...|2025-02-28 15:31:...|
+----------+--------+--------------------+--------------------+



In [56]:
def extract_from_csv(spark: SparkSession, path: str, file_name):
    """extract data from CSV file."""
    current_timestamp = datetime.now()
    
    try:
        print(f"Extracting CSV: {file_name}")
        
        df = spark.read.csv(
            path + file_name,
            header=True,
            inferSchema=True
        )
        
        print(f"Finished extracting data from {file_name}: {df.count()} rows, {len(df.columns)} columns")
        log_message = spark.sparkContext \
            .parallelize([("extraction", "source", "failed", file_name, current_timestamp)]) \
            .toDF(["step", "component", "status", "table_name", "etl_date"])
        return df
        
    except Exception as e:
        print(f"Error extracting data from table {table_name}: {e}")

        log_message = spark.sparkContext \
            .parallelize([("extraction", "source", "failed", file_name, current_timestamp, str(e))]) \
            .toDF(["step", "component", "status", "table_name", "etl_date", "error_msg"])

    finally:
        load_log_msg(spark=spark, log_msg=log_message)

In [57]:
df_csv = extract_from_csv(spark=spark, path="data/", file_name="new_bank_transaction.csv")

Extracting CSV: new_bank_transaction.csv
Finished extracting data from new_bank_transaction.csv: 1048567 rows, 9 columns
finished writing log..


In [19]:
df_csv.printSchema()

root
 |-- TransactionID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- CustomerDOB: string (nullable = true)
 |-- CustGender: string (nullable = true)
 |-- CustLocation: string (nullable = true)
 |-- CustAccountBalance: double (nullable = true)
 |-- TransactionDate: string (nullable = true)
 |-- TransactionTime: integer (nullable = true)
 |-- TransactionAmount (INR): double (nullable = true)



In [22]:
df_marital_status.printSchema()

root
 |-- marital_id: integer (nullable = true)
 |-- value: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)



In [23]:
# number of rows and column
# name of columns
# data type for each column
# missing value percentage

'''Before developing the data pipeline, the first step is to explore and understand the data. 
This includes performing data profiling to review the structure, 
identify missing values, check data types, and detect inconsistencies. 
By doing this early, we can better determine what transformations are needed to clean, 
standardize, and prepare the data for further analysis. 
This process helps ensure that the pipeline is built based on actual data conditions'''

def profiling_table(data, table_name: str):
    report = {}

    
    

profiling_table(data=df_marital_status, table_name="marital_status")


 Table: marital_status
 Rows   : 3
 Columns: 4


In [26]:
df_marketing_campaign_deposit.printSchema()

root
 |-- loan_data_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital_id: integer (nullable = true)
 |-- education_id: integer (nullable = true)
 |-- default: boolean (nullable = true)
 |-- balance: string (nullable = true)
 |-- housing: boolean (nullable = true)
 |-- loan: boolean (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- subscribed_deposit: boolean (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)



In [61]:
from pyspark.sql.functions import regexp_replace, floor
from datetime import datetime
# from src.helper_function.helper import etl_log


def transform_marketing_campaign(spark: SparkSession, df):
    try:
        current_timestamp = datetime.now()
        column_to_rename = {
            "pdays":"days_since_last_campaign",
            "previous":"previous_campaign_contacts",
            "poutcome":"previous_campaign_outcome"
        }

        df = df.withColumn('balance', F.regexp_replace(df['balance'], r"\$", " "))
        df = df.withColumn('balance', df['balance'].cast('int'))

        df = df.withColumn('duration_in_year', F.floor(df['duration'] / 365).cast('int'))

        df = df.withColumnsRenamed(column_to_rename)

        print("finished transforming table.")

        log_message = spark.sparkContext \
            .parallelize([("transformation", "source", "success", "marketing_campaign_deposit", current_timestamp)]) \
            .toDF(["step", "component", "status", "table_name", "etl_date"])
        
        return df
    except Exception as e:
        print("error transforming data: ", str(e))
        
        log_message = spark.sparkContext \
            .parallelize([("transformation", "source", "failed", "marketing_campaign_deposit", current_timestamp, str(e))]) \
            .toDF(["step", "component", "status", "table_name", "etl_date", "error_msg"])
    finally:
        load_log_msg(spark=spark, log_msg=log_message)    
    

In [63]:
df = transform_marketing_campaign(spark=spark, df=df_marketing_campaign_deposit)

finished transforming table.
finished writing log..
root
 |-- loan_data_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital_id: integer (nullable = true)
 |-- education_id: integer (nullable = true)
 |-- default: boolean (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: boolean (nullable = true)
 |-- loan: boolean (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- days_since_last_campaign: integer (nullable = true)
 |-- previous_campaign_contacts: integer (nullable = true)
 |-- previous_campaign_outcome: string (nullable = true)
 |-- subscribed_deposit: boolean (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- duration_in_year: integer (nullable = true)



In [30]:
df_csv.printSchema()

root
 |-- TransactionID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- CustomerDOB: string (nullable = true)
 |-- CustGender: string (nullable = true)
 |-- CustLocation: string (nullable = true)
 |-- CustAccountBalance: double (nullable = true)
 |-- TransactionDate: string (nullable = true)
 |-- TransactionTime: integer (nullable = true)
 |-- TransactionAmount (INR): double (nullable = true)



In [ ]:
# Source - https://stackoverflow.com/a/42537653
# Posted by Grr, modified by community. See post 'Timeline' for change history
# Retrieved 2026-05-26, License - CC BY-SA 3.0

from pyspark.sql import functions as F
df.withColumn('device_id', F.when(col('device')=='desktop', 1).when(col('device')=='mobile', 2).otherwise(None))


In [38]:
df_csv.select("CustomerDOB").distinct().show()

+-----------+
|CustomerDOB|
+-----------+
|    31/7/84|
|    17/3/87|
|   11/10/89|
|    4/12/95|
|    30/6/96|
|    16/4/88|
|    17/6/87|
|    30/6/86|
|    22/9/89|
|    3/10/90|
|   16/12/94|
|    15/1/76|
|     9/7/92|
|    24/2/88|
|    14/3/77|
|     9/2/92|
|    28/6/85|
|   22/10/82|
|    1/11/90|
|   21/12/81|
+-----------+
only showing top 20 rows



In [68]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit, to_date, date_format, year, round, from_unixtime, make_date, year, month, dayofmonth

def transform_customer(spark: SparkSession, df):
    try:
        current_timestamp = datetime.now()
        column_to_rename = {
            "CustomerID":"customer_id",
            "CustomerDOB":"birth_date",
            "CustGender":"gender",
            "CustLocation":"location",
            "CustAccountBalance":"account_balance"
        }

        df = df.withColumnsRenamed(column_to_rename)
        
        df = df.withColumn("birth_date", to_date(col("birth_date"), "d/M/yy"))
        df = df.withColumn("birth_date",
                when(
                    year(col("birth_date")) > 2025,
                    make_date(
                        year(col("birth_date")) - 100,
                        month(col("birth_date")),
                        dayofmonth(col("birth_date"))
                    )
                ).otherwise(col("birth_date"))
            )
        
        df = df.withColumn('gender', F.when(col('gender')=='M', 'Male').when(col('gender')=='F', 'Female').otherwise('Other'))

        df = df.withColumn("account_balance", round(df["account_balance"].cast("float")))
        
        df = df.select(
            'customer_id', 
            'birth_date', 
            'gender', 
            'location', 
            'account_balance' 
        )
        
        log_message = spark.sparkContext \
            .parallelize([("transformation", "source", "success", "customers", current_timestamp)]) \
            .toDF(["step", "component", "status", "table_name", "etl_date"])
        
        return df
        
    except Exception as e:
        print("error transforming data: ", str(e))

        log_message = spark.sparkContext \
            .parallelize([("transformation", "source", "failed", "customers", current_timestamp, str(e))]) \
            .toDF(["step", "component", "status", "table_name", "etl_date", "error_msg"])
    finally:
        load_log_msg(spark=spark,log_msg=log_message)

In [69]:
cust_transformed = transform_customer(spark=spark, df=df_csv)

finished writing log..


In [70]:
cust_transformed.orderBy(col("birth_date").desc()).show()

+-----------+----------+------+----------+---------------+
|customer_id|birth_date|gender|  location|account_balance|
+-----------+----------+------+----------+---------------+
|   C3922921|2025-05-06|Female|TRIVANDRUM|       604464.0|
|   C3452032|2024-02-09|Female|    MUMBAI|        60555.0|
|   C2951578|2024-02-02|Female| HYDERABAD|      4823173.0|
|   C2751560|2024-02-02|Female| HYDERABAD|      4823173.0|
|   C6043541|2023-08-06|  Male|   CHENNAI|        20421.0|
|   C1943541|2023-08-06|  Male|   CHENNAI|        20421.0|
|   C6843517|2023-08-06|  Male|   CHENNAI|        20421.0|
|   C8243524|2023-08-06|  Male|   CHENNAI|        20421.0|
|   C1043589|2023-08-06|  Male|   CHENNAI|        20421.0|
|   C8143576|2023-08-06|  Male|   CHENNAI|        20421.0|
|   C6842464|2022-10-20|  Male|    KANPUR|         6832.0|
|   C7242435|2022-10-20|  Male|    KANPUR|         6832.0|
|   C7642474|2022-10-20|  Male|    KANPUR|         6832.0|
|   C7017646|2021-01-19|  Male| NEW DELHI|      1125922.

In [43]:
cust_transformed.show()

+-----------+----------+------+-----------+---------------+
|customer_id|birth_date|gender|   location|account_balance|
+-----------+----------+------+-----------+---------------+
|   C1010028|1988-08-25|Female|      DELHI|       296828.0|
|   C1010035|1992-03-02|  Male|     MUMBAI|         7284.0|
| C1010035_2|1980-06-09|  Male|NAVI MUMBAI|       378013.0|
|   C1010036|1996-02-26|  Male|    GURGAON|       355430.0|
|   C1010041|1993-09-06|Female|      DELHI|        34119.0|
| C1010041_2|1975-09-14|Female|      NOIDA|       746732.0|
| C1010041_3|1992-07-13|Female|      LOHIT|         1291.0|
|   C1010064|1988-09-19|  Male|      DELHI|            2.0|
|   C1010065|1985-02-16|  Male|  BARABANKI|        18819.0|
|   C1010071|1985-09-14|Female|  NEW DELHI|       329868.0|
|   C1010074|1985-09-14|Female|  NEW DELHI|       329868.0|
|   C1010078|1983-09-16|Female|     MUMBAI|      1421750.0|
|   C1010129|1992-09-15|  Male|    KOLKATA|        27340.0|
| C1010129_2|1997-12-22|  Male|     NAGP

In [71]:
def transform_transaction(spark: SparkSession, df):
    try:
        current_timestamp = datetime.now()
        column_to_rename = {
            'TransactionID':'transaction_id',
            'CustomerID':'customer_id',
            'TransactionDate':'transaction_date',
            'TransactionTime':'transaction_time',
            'TransactionAmount (INR)':'transaction_amount'
        }
        df = df.withColumnsRenamed(column_to_rename)

        df = df.withColumn("transaction_date", to_date(col("transaction_date"), "d/M/yy"))
        df = df.withColumn("transaction_date",
                when(
                    year(col("transaction_date")) > 2025,
                    make_date(
                        year(col("transaction_date")) - 100,
                        month(col("transaction_date")),
                        dayofmonth(col("transaction_date"))
                    )
                ).otherwise(col("transaction_date"))
            )

        df = df.withColumn("transaction_time",from_unixtime(col("transaction_time").cast("int"), "HH:mm:ss"))

        df = df.withColumn("transaction_amount", round(df["transaction_amount"].cast("float")))

        df = df.select(
            'transaction_id', 
            'customer_id', 
            'transaction_date', 
            'transaction_time', 
            'transaction_amount' 
        )
        print('Finished transforming table transaction')
        log_message = spark.sparkContext \
            .parallelize([("transformation", "source", "success", "transactions", current_timestamp)]) \
            .toDF(["step", "component", "status", "table_name", "etl_date"])
        
        return df
    except Exception as e:
        print("error transforming data: ", str(e))
        log_message = spark.sparkContext \
            .parallelize([("transformation", "source", "failed", "transactions", current_timestamp, str(e))]) \
            .toDF(["step", "component", "status", "table_name", "etl_date", "error_msg"])
    finally:
        # pass
        load_log_msg(spark=spark, log_msg=log_message)

In [73]:
transformed_transaction = transform_transaction(spark=spark,df=df_csv)

Finished transforming table transaction
finished writing log..


In [52]:
transformed_transaction.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- transaction_time: string (nullable = true)
 |-- transaction_amount: float (nullable = true)



In [53]:
transformed_transaction.show()

+--------------+-----------+----------------+----------------+------------------+
|transaction_id|customer_id|transaction_date|transaction_time|transaction_amount|
+--------------+-----------+----------------+----------------+------------------+
|       T642232|   C1010028|      2016-08-29|        02:26:52|             557.0|
|        T87414|   C1010035|      2016-08-01|        07:05:17|              50.0|
|       T560676| C1010035_2|      2016-08-27|        03:23:31|             700.0|
|       T610204|   C1010036|      2016-08-26|        02:26:43|             208.0|
|       T957663|   C1010041|      2016-09-10|        21:08:53|           14500.0|
|       T113533| C1010041_2|      2016-08-06|        23:38:57|            2397.0|
|       T888200| C1010041_3|      2016-09-07|        00:24:19|              20.0|
|       T152889|   C1010064|      2016-08-05|        07:45:05|            3000.0|
|       T670148|   C1010065|      2016-08-28|        21:03:51|             500.0|
|       T601310|

In [27]:
df_marital_status.show()

+----------+--------+--------------------+--------------------+
|marital_id|   value|          created_at|          updated_at|
+----------+--------+--------------------+--------------------+
|         1| married|2025-02-28 15:31:...|2025-02-28 15:31:...|
|         2|  single|2025-02-28 15:31:...|2025-02-28 15:31:...|
|         3|divorced|2025-02-28 15:31:...|2025-02-28 15:31:...|
+----------+--------+--------------------+--------------------+



In [36]:
def load_to_dwh(spark: SparkSession, df, table_name: str):
    
    DWH_JDBC_URL = f"jdbc:postgresql://data_warehouse_container:5432/data_warehouse"
    DWH_POSTGRES_USER = 'postgres'
    DWH_POSTGRES_PASSWORD = 'postgres'
    
    current_timestamp = datetime.now()
    
    try:
        # truncate table
        connection = spark._jvm.java.sql.DriverManager.getConnection(
                DWH_JDBC_URL, DWH_POSTGRES_USER, DWH_POSTGRES_PASSWORD
            )
        statement = connection.createStatement()
        statement.executeUpdate(f"TRUNCATE TABLE {table_name} CASCADE")
        connection.close()
        
        print(f"Success truncating table: {table_name}")

        # start loading data
        print(f"start loading table: {table_name}")
        
        df.write.jdbc(
                url=DWH_JDBC_URL,
                table=table_name,
                mode="append",
                properties={
                    "user": DWH_POSTGRES_USER,
                    "password": DWH_POSTGRES_PASSWORD
                }
            )
        print(f"finished loading table: {table_name}")
        
        log_message = spark.sparkContext \
            .parallelize([("loading", "warehouse", "success", table_name, current_timestamp)]) \
            .toDF(["step", "component", "status", "table_name", "etl_date"])
        
    except Exception as e:
        print(f"Load process failed: {e}")

        log_message = spark.sparkContext\
            .parallelize([("loading", "warehouse", "failed", table_name, current_timestamp, str(e))])\
            .toDF(['step', 'component', 'status', 'table_name', 'etl_date', 'error_msg'])
        
    finally:
        load_log_msg(spark=spark, log_msg=log_message)
        
        
        
    

In [37]:
load_to_dwh(spark=spark, df=df_marital_status, table_name='marital_status')

Success truncating table: marital_status
start loading table: marital_status
finished loading table: marital_status
finished writing log..
+-------+---------+-------+--------------+--------------------+
|   step|component| status|    table_name|            etl_date|
+-------+---------+-------+--------------+--------------------+
|loading|warehouse|success|marital_status|2026-05-30 08:19:...|
+-------+---------+-------+--------------+--------------------+



In [75]:
load_to_dwh(spark=spark, df=cust_transformed, table_name='customers')

Success truncating table: customers
start loading table: customers
finished loading table: customers
finished writing log..
+-------+---------+-------+----------+--------------------+
|   step|component| status|table_name|            etl_date|
+-------+---------+-------+----------+--------------------+
|loading|warehouse|success| customers|2026-05-30 14:58:...|
+-------+---------+-------+----------+--------------------+

